<a href="https://colab.research.google.com/github/Waldeniralexandre/sicop-mvp-puc-2026/blob/main/SICOP_MVP_ML_Analytics_20261.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP — Machine Learning & Analytics

**Nome:** Waldenir Alexandre da Silva Cruz

**Matrícula:** 4052026000470

**Curso ou programa:** Ciência de Dados & Analytics

**Disciplina:** Machine Learning & Analytics

**Turma:** 1

**Professor:** Dra. Tatiana Escovedo

**Data de entrega:** 05/07/2026

**Dataset:** SICOP — registros de ocorrências em `data/raw/MVP_wascBD_Acidentes.xlsx`, planilha `BD_Acidentes`.

**Tipo de problema:** classificação binária supervisionada.

Este notebook reconstrói o MVP de Machine Learning e Analytics do projeto SICOP em conformidade com o template oficial da PUC-Rio. O objetivo é estimar, após o registro inicial da ocorrência e antes da avaliação técnica de risco, a probabilidade de a ocorrência pertencer às categorias Alto ou Crítico.


## Checklist do MVP

| Item | Status |
|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | preenchido |
| Dataset descrito, com fonte, atributos e restrições | preenchido |
| Análise exploratória do treino documentada | preenchido |
| Divisão temporal preservada | preenchido |
| Pipeline e features congelados | preenchido |
| Baseline e modelos candidatos documentados | preenchido |
| Avaliação final carregada de artefatos versionados | preenchido |
| Limitações e rastreabilidade registradas | preenchido |


# 1. Definição do problema

## 1.1 Descrição do problema

O SICOP organiza ocorrências de segurança e operação. O problema de negócio é apoiar a triagem técnica após o registro inicial, priorizando ocorrências que podem ser enquadradas posteriormente como críticas. O problema de Machine Learning é estimar a probabilidade de uma ocorrência individual pertencer às categorias Alto ou Crítico, sem usar informações posteriores à avaliação técnica.

Unidade de análise: uma ocorrência individual registrada na base.

Momento da predição: após o registro inicial da ocorrência e antes da avaliação técnica de risco.

Target de modelagem: `Evento_Crítico_Target_Modelagem`.

Classe positiva: 1, correspondente a Alto ou Crítico.

Uso pretendido: apoio à priorização probabilística e à triagem humana. O modelo não substitui avaliação técnica humana e não prevê a ocorrência de um acidente antes que ele aconteça.


## 1.2 Objetivo do MVP

Construir e avaliar um pipeline reprodutível de classificação binária para estimar se uma ocorrência registrada será classificada como evento crítico, comparando o candidato selecionado contra um baseline e preservando a separação temporal entre treino, validação e teste.

O objetivo acadêmico é demonstrar um fluxo metodologicamente válido, rastreável, sem vazamento conhecido e adequado ao template da PUC-Rio.


## 1.3 Tipo de problema

Este MVP trata de aprendizado supervisionado — classificação binária. O target reconstruído é `Evento_Crítico_Target_Modelagem`, com o seguinte mapeamento:

- Baixo e Médio: 0;
- Alto e Crítico: 1.

A saída probabilística é interpretada como apoio à priorização. A decisão binária operacional não é aprovada.


## 1.4 Premissas, hipóteses e critérios de sucesso

Hipóteses:

1. As informações estruturadas disponíveis no registro inicial possuem sinal preditivo para priorização probabilística.
2. Um modelo candidato deve superar o baseline em métricas probabilísticas, especialmente Average Precision e ROC-AUC.
3. A identificação de eventos críticos é prioritária, mas o threshold binário precisa ser criticamente avaliado.

Critérios de sucesso acadêmico:

- pipeline reproduzível e rastreável;
- divisão temporal preservada;
- ausência de ajuste com o teste;
- comparação com baseline;
- interpretação clara das limitações;
- natureza experimental do modelo explicitada.


# 2. Ambiente, bibliotecas e reprodutibilidade

O notebook usa caminhos relativos, `random_state=42`, bibliotecas padrão de análise tabular e os artefatos versionados do projeto. A avaliação final do teste não é reexecutada neste notebook: seus resultados são carregados de tabelas, gráficos e manifesto já produzidos e congelados.


In [1]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    # Fallback for Colab environment where the strict project structure might not be present initially
    print("AVISO: Marcadores da raiz do projeto (AGENTS.md, diretório notebooks) não encontrados.")
    print("Definindo a raiz do projeto para o diretório de trabalho atual (e.g., /content/).")
    return Path.cwd() # Assume current working directory as project root in Colab

ROOT = find_project_root()
print(f"Raiz do projeto: {ROOT.name}")
print(f"random_state={SEED}")

AVISO: Marcadores da raiz do projeto (AGENTS.md, diretório notebooks) não encontrados.
Definindo a raiz do projeto para o diretório de trabalho atual (e.g., /content/).
Raiz do projeto: content
random_state=42


## 2.1 Dependências adicionais

Não são necessárias dependências adicionais. O notebook utiliza pandas, NumPy, matplotlib e recursos de exibição do IPython. Não há dependência de internet, Google Drive ou arquivos externos ao pacote.


In [2]:
def require_file(relative_path):
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {relative_path}")
    return path

def read_csv(relative_path):
    return pd.read_csv(require_file(relative_path), encoding="utf-8-sig")

def read_json(relative_path):
    with require_file(relative_path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def show_image(relative_path):
    path = require_file(relative_path)
    display(Image(filename=str(path)))

def display_table(df, rows=10):
    display(df.head(rows))


## 2.2 Funções auxiliares

As funções acima validam a existência de arquivos, carregam CSV/JSON por caminhos relativos e exibem tabelas ou gráficos versionados. Elas não treinam modelo, não recalculam a avaliação final e não acessam recursos externos.


In [3]:
PATHS = {
    "raw": Path("data/raw/MVP_wascBD_Acidentes.xlsx"),
    "train": Path("data/processed/SICOP_train.csv"),
    "validation": Path("data/processed/SICOP_validation.csv"),
    "test": Path("data/processed/SICOP_test.csv"),
    "train_clean": Path("data/processed/SICOP_train_limpo.csv"),
    "validation_clean": Path("data/processed/SICOP_validation_limpo.csv"),
    "test_clean": Path("data/processed/SICOP_test_limpo.csv"),
    "manifest15": Path("outputs/metrics/15_manifest_modelagem_validacao.json"),
    "manifest17": Path("outputs/metrics/17_manifest_avaliacao_final.json"),
}
for label, relative in PATHS.items():
    require_file(relative)
print("Arquivos obrigatórios localizados.")


FileNotFoundError: Arquivo obrigatório não encontrado: data/raw/MVP_wascBD_Acidentes.xlsx

# 3. Seleção e carga dos dados

## 3.1 Fonte dos dados

A base original é `data/raw/MVP_wascBD_Acidentes.xlsx`, planilha `BD_Acidentes`, preservada sem alteração. A base contém 1.718 registros e 20 colunas antes das derivações controladas. Os dados são registros históricos de ocorrências e possuem limitações de qualidade, disponibilidade temporal, categorias raras e possível variação operacional ao longo do tempo.


## 3.2 Carga dos dados

O notebook carrega as bases processadas e artefatos versionados. A base bruta é referenciada para rastreabilidade, mas a modelagem final usa as bases derivadas já materializadas e testadas.


In [ ]:
train = read_csv(PATHS["train"])
validation = read_csv(PATHS["validation"])
test = read_csv(PATHS["test"])
train_clean = read_csv(PATHS["train_clean"])
validation_clean = read_csv(PATHS["validation_clean"])
test_clean = read_csv(PATHS["test_clean"])

manifest15 = read_json(PATHS["manifest15"])
manifest17 = read_json(PATHS["manifest17"])

print("Treino:", train.shape)
print("Validação:", validation.shape)
print("Teste:", test.shape)
print("Treino limpo:", train_clean.shape)
print("Validação limpa:", validation_clean.shape)
print("Teste limpo:", test_clean.shape)


## 3.3 Visão geral do dataset

A base oficial possui 1.718 registros. A auditoria identificou 19 linhas duplicadas além da primeira ocorrência. Essas linhas foram preservadas e controladas por `Grupo_Duplicata_Split`, sem remoção indevida e sem grupos atravessando conjuntos.


In [ ]:
overview = pd.DataFrame(
    [
        {"conjunto": "treino", "linhas": len(train), "colunas": train.shape[1]},
        {"conjunto": "validação", "linhas": len(validation), "colunas": validation.shape[1]},
        {"conjunto": "teste", "linhas": len(test), "colunas": test.shape[1]},
        {"conjunto": "treino_limpo", "linhas": len(train_clean), "colunas": train_clean.shape[1]},
        {"conjunto": "validacao_limpa", "linhas": len(validation_clean), "colunas": validation_clean.shape[1]},
        {"conjunto": "teste_limpo", "linhas": len(test_clean), "colunas": test_clean.shape[1]},
    ]
)
display(overview)


In [ ]:
target_col = "Evento_Crítico_Target_Modelagem"
target_summary = pd.DataFrame(
    {
        "treino": train_clean[target_col].value_counts().sort_index(),
        "validacao": validation_clean[target_col].value_counts().sort_index(),
        "teste": test_clean[target_col].value_counts().sort_index(),
    }
).fillna(0).astype(int)
display(target_summary)


In [ ]:
missing_features = train_clean.isna().sum().to_frame("ausentes_treino_limpo")
display(missing_features)


In [ ]:
duplicate_summary = pd.DataFrame(
    [
        {"conjunto": "treino_limpo", "duplicatas_preservadas": int(train_clean.duplicated().sum())},
        {"conjunto": "validacao_limpa", "duplicatas_preservadas": int(validation_clean.duplicated().sum())},
        {"conjunto": "teste_limpo", "duplicatas_preservadas": int(test_clean.duplicated().sum())},
    ]
)
display(duplicate_summary)


## 3.4 Dicionário de dados

As features congeladas são majoritariamente categóricas e representam informações estruturadas disponíveis antes da avaliação técnica de risco. `Ano` é metadado temporal, `Grupo_Duplicata_Split` é metadado técnico, e `Evento_Crítico_Target_Modelagem` é o target.

| Coluna | Papel | Observações |
|---|---|---|
| Classificação | feature categórica | candidata preservada após análise de sensibilidade |
| Mês | feature categórica | derivada temporal tratada como categoria |
| Dia | feature categórica | derivada temporal tratada como categoria |
| Hora | feature categórica | normalização textual determinística |
| Dia da Semana | feature categórica | reconstruído por calendário real |
| Estado | feature categórica | unidade federativa |
| Segmento | feature categórica | categoria operacional |
| Faixas de experiência | feature categórica | harmonização determinística |
| Compromissos (Regra de Ouro) | feature categórica | categoria operacional |
| Ano | metadado | não feature |
| Grupo_Duplicata_Split | metadado | não feature |
| Evento_Crítico_Target_Modelagem | target | 0 para Baixo/Médio; 1 para Alto/Crítico |


# 4. Análise exploratória dos dados

A análise exploratória com risco de influenciar decisões de modelagem foi restrita ao conjunto de treino. O conjunto de validação e o teste não foram usados para adaptar regras exploratórias, escolher features ou alterar thresholds.


In [ ]:
FEATURES = [
    "Classificação",
    "Mês",
    "Dia",
    "Hora",
    "Dia da Semana",
    "Estado",
    "Segmento",
    "Faixas de experiência",
    "Compromissos (Regra de Ouro)",
]
target_counts_train = train_clean[target_col].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3))
target_counts_train.plot(kind="bar", ax=ax)
ax.set_title("Distribuição do target no treino")
ax.set_xlabel("Classe")
ax.set_ylabel("Registros")
plt.tight_layout()
plt.show()


In [ ]:
temporal_train = train_clean.groupby("Ano").size().reset_index(name="registros")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(temporal_train["Ano"], temporal_train["registros"], marker="o")
ax.set_title("Evolução temporal no treino")
ax.set_xlabel("Ano")
ax.set_ylabel("Registros")
plt.tight_layout()
plt.show()


## 4.1 Síntese da análise exploratória

O treino possui 1.126 registros, com 903 ocorrências da classe 0 e 223 da classe 1. As features são categóricas, com categorias raras preservadas. A limpeza determinística removeu inconsistências textuais, reconstruiu `Dia da Semana` por calendário real e manteve ausências controladas como `<AUSENTE>` quando aplicável.


# 5. Preparação dos dados e divisão treino/teste

As nove features congeladas são:

- Classificação
- Mês
- Dia
- Hora
- Dia da Semana
- Estado
- Segmento
- Faixas de experiência
- Compromissos (Regra de Ouro)

`Ano`, `Grupo_Duplicata_Split` e `Evento_Crítico_Target_Modelagem` não são features.


In [ ]:
split_summary = pd.DataFrame(
    [
        {"conjunto": "treino", "periodo": "datas anteriores a 2025-01-01", "linhas": 1126, "classe_0": 903, "classe_1": 223},
        {"conjunto": "validação", "periodo": "2025-01-01 <= data < 2025-06-01", "linhas": 229, "classe_0": 188, "classe_1": 41},
        {"conjunto": "teste", "periodo": "data >= 2025-06-01", "linhas": 363, "classe_0": 282, "classe_1": 81},
    ]
)
display(split_summary)


In [ ]:
assert len(train_clean) == 1126
assert len(validation_clean) == 229
assert len(test_clean) == 363
assert int((train_clean[target_col] == 0).sum()) == 903
assert int((train_clean[target_col] == 1).sum()) == 223
assert int((validation_clean[target_col] == 0).sum()) == 188
assert int((validation_clean[target_col] == 1).sum()) == 41
assert int((test_clean[target_col] == 0).sum()) == 282
assert int((test_clean[target_col] == 1).sum()) == 81
print("Split temporal validado pelos artefatos carregados.")


## 5.1 Justificativa da divisão

A divisão temporal foi adotada para simular uso futuro do modelo e evitar mistura aleatória de registros de períodos posteriores no treino. O embaralhamento seria inadequado porque permitiria estimar desempenho com informação temporalmente misturada. Grupos duplicados foram protegidos por `Grupo_Duplicata_Split`, sem grupos cruzando treino, validação e teste. O teste foi preservado até a avaliação final única.


# 6. Pré-processamento e pipeline

Todas as features congeladas são tratadas como categóricas. O pipeline usa `OneHotEncoder(handle_unknown="ignore")`, ajustado apenas no treino de cada fold ou no treino completo antes da validação, preservando categorias desconhecidas sem aprender regras com validação ou teste.


In [ ]:
preprocessamento = pd.DataFrame(
    [
        {"etapa": "limpeza textual", "descricao": "NFKC, NBSP, colapso de espaços e strip"},
        {"etapa": "Hora", "descricao": "normalização determinística para domínio fechado"},
        {"etapa": "Dia da Semana", "descricao": "reconstrução por Ano, Mês e Dia"},
        {"etapa": "Faixas de experiência", "descricao": "mapeamento determinístico; ausentes como <AUSENTE>"},
        {"etapa": "encoding", "descricao": 'OneHotEncoder com handle_unknown="ignore"'},
    ]
)
display(preprocessamento)


## 6.1 Decisões de pré-processamento

As transformações foram definidas no treino, validadas fora da amostra e aplicadas ao teste sem adaptação. Não houve imputação por moda, seleção automática usando teste, escalonamento numérico relevante ou aprendizado de vocabulário com validação/teste. Essa estratégia reduz risco de vazamento.


# 7. Baseline e modelos candidatos

Baseline: `DummyClassifier(strategy="prior")`.

Modelos candidatos:

- Regressão Logística;
- Árvore de Decisão;
- Random Forest.

Foram avaliadas 84 configurações não baseline em três folds temporais expansivos, sem uso do teste final.


In [ ]:
folds = pd.DataFrame(
    [
        {"fold": 1, "treino": "2020-2021", "avaliacao": "2022", "linhas_treino": 139, "linhas_avaliacao": 164},
        {"fold": 2, "treino": "2020-2022", "avaliacao": "2023", "linhas_treino": 303, "linhas_avaliacao": 338},
        {"fold": 3, "treino": "2020-2023", "avaliacao": "2024", "linhas_treino": 641, "linhas_avaliacao": 485},
    ]
)
display(folds)


## 7.1 Justificativa dos modelos

O baseline estabelece referência simples. Regressão Logística oferece interpretabilidade, Árvore de Decisão representa alternativa não linear simples e Random Forest permite capturar interações categóricas após encoding. A seleção foi feita apenas com treino e validação interna temporal.


# 8. Treinamento e avaliação inicial

O treinamento experimental ocorreu no Passo 15, com treino e validação temporal. Este notebook não retreina modelos; ele documenta e carrega evidências versionadas.


In [ ]:
validation_results = read_csv(Path("outputs/tables/15_resultados_validacao.csv"))
validation_confusion = read_csv(Path("outputs/tables/15_matrizes_confusao_validacao.csv"))
display_table(validation_results, rows=10)
display_table(validation_confusion, rows=10)


## 8.1 Análise dos resultados iniciais

O candidato selecionado superou o baseline em Average Precision e ROC-AUC na validação. As métricas preservadas foram: Average Precision 0.49395748713, ROC-AUC 0.743383497665, Brier Score 0.135832612908, recall 1.0, precision 0.17903930131, F2 0.521628498728, balanced accuracy 0.5, TN=0, FP=188, FN=0 e TP=41.

O threshold congelado classificou todos os 229 registros de validação como positivos. Isso preserva utilidade probabilística, mas limita a interpretação binária.


# 9. Validação e otimização de hiperparâmetros

A otimização foi limitada aos folds temporais internos do treino. Não foi reaberta após a validação final e não usou o conjunto de teste. O threshold congelado foi escolhido exclusivamente com previsões out-of-fold anteriores ao teste final.


In [ ]:
selected_configs = read_csv(Path("outputs/tables/15_configuracoes_selecionadas.csv"))
threshold_search = read_csv(Path("outputs/tables/15_busca_threshold.csv"))
display_table(selected_configs, rows=10)
display_table(threshold_search, rows=10)


## 9.1 Discussão da otimização

Modelo selecionado:

- config_id: `com_classificacao|random_forest|class_weight=None|max_depth=5|min_samples_leaf=10`;
- família: Random Forest;
- class_weight: None;
- max_depth: 5;
- max_features: sqrt;
- min_samples_leaf: 10;
- n_estimators: 300;
- n_jobs: 1;
- random_state: 42;
- threshold congelado: 0.108864504929.


# 10. Avaliação final no conjunto de teste

A avaliação final do teste não é reexecutada neste notebook. Os resultados abaixo são carregados de `outputs/tables/17_resultados_teste.csv`, `outputs/tables/17_matrizes_confusao_teste.csv`, `outputs/tables/17_comparacao_validacao_teste.csv` e `outputs/metrics/17_manifest_avaliacao_final.json`.


In [ ]:
test_results = read_csv(Path("outputs/tables/17_resultados_teste.csv"))
test_confusion = read_csv(Path("outputs/tables/17_matrizes_confusao_teste.csv"))
validation_test_comparison = read_csv(Path("outputs/tables/17_comparacao_validacao_teste.csv"))
score_distribution = read_csv(Path("outputs/tables/17_distribuicao_scores_teste.csv"))

display(test_results)
display(test_confusion)


## 10.1 Análise de erros e limitações

Candidato no threshold 0,5:

- Average Precision: 0.37299847818328125;
- ROC-AUC: 0.6777865335784957;
- Brier Score: 0.16528189652763325;
- TN: 282;
- FP: 0;
- FN: 81;
- TP: 0;
- previsões positivas: 0;
- previsões negativas: 363.

Candidato no threshold congelado 0.108864504929:

- accuracy: 0.22589531680440772;
- balanced accuracy: 0.50177304964539;
- precision: 0.22375690607734808;
- recall: 1.0;
- F1: 0.3656884875846501;
- F2: 0.5903790087463557;
- TN: 1;
- FP: 281;
- FN: 0;
- TP: 81;
- previsões positivas: 362;
- previsões negativas: 1.

Baseline no teste:

- Average Precision: 0.2231404958677686;
- ROC-AUC: 0.5;
- Brier Score: 0.17397853960169055.


# 11. Comparação final dos modelos

A comparação validação versus teste mostra queda de Average Precision, queda de ROC-AUC e aumento do Brier Score para o candidato. A interpretação deve ser prudente: há sinal de generalização temporal, mas com redução de desempenho e ausência de qualquer ajuste decorrente do resultado do teste.


In [ ]:
display_table(validation_test_comparison, rows=20)


# 12. Boas práticas e rastreabilidade

Reprodutibilidade:

- Python 3.12.10;
- NumPy 2.5.1;
- pandas 3.0.3;
- scikit-learn 1.9.0;
- matplotlib 3.11.0;
- joblib 1.5.3;
- `random_state=42`;
- caminhos relativos;
- manifestos com hashes;
- commit do pré-registro: e6cc91d;
- commit da avaliação final: df6d5a7;
- teste não usado para ajuste.

Modelo, features, hiperparâmetros e threshold permaneceram congelados após o teste.


# 13. Conclusão

O pipeline é metodologicamente válido para o objetivo acadêmico do MVP, sem vazamento conhecido e com rastreabilidade entre dados, código, artefatos e documentos. O candidato supera o baseline em métricas probabilísticas no teste, com Average Precision superior à prevalência, ROC-AUC superior a 0,5, ROC-AUC de 0.5 no baseline e Brier Score inferior ao baseline.

Houve redução de desempenho no teste temporal em relação à validação. O modelo possui potencial para priorização probabilística, mas os thresholds avaliados não sustentam decisão binária operacional. O objetivo acadêmico do MVP foi atendido, e novos dados, validações externas e decisões operacionais seriam necessários antes de qualquer uso real.


# 14. Salvamento de artefatos

Os artefatos já foram salvos por scripts oficiais versionados. Este notebook apenas carrega:

- `outputs/models/15_candidate_pipeline.joblib`;
- `outputs/metrics/15_manifest_modelagem_validacao.json`;
- `outputs/metrics/17_manifest_avaliacao_final.json`;
- tabelas finais em `outputs/tables`;
- gráficos finais em `outputs/plots`;
- síntese final em `outputs/text/17_sintese_para_notebook.md`.

Os artefatos destinam-se ao notebook oficial e ao pacote final de entrega.


In [ ]:
print("Gráfico Precision-Recall final")
show_image(Path("outputs/plots/17_precision_recall_teste.png"))


In [ ]:
print("Gráfico ROC final")
show_image(Path("outputs/plots/17_roc_teste.png"))


In [ ]:
print("Distribuição dos scores no teste")
show_image(Path("outputs/plots/17_distribuicao_scores_teste.png"))


# 15. Apêndice: referências, limitações e interpretação

## Interpretação probabilística

- o candidato supera o baseline;
- Average Precision é superior à prevalência;
- ROC-AUC é superior a 0,5;
- Brier Score é inferior ao baseline;
- existe sinal de ordenação probabilística.

## Interpretação binária

- threshold 0,5 não identifica positivos;
- threshold congelado identifica todos os positivos;
- threshold congelado gera 281 falsos positivos;
- 362 de 363 registros são classificados como positivos;
- uso binário operacional é inadequado.

## Limitações

Incluem volume e abrangência dos dados, mudanças temporais, categorias desconhecidas, dependência da qualidade do registro, variáveis majoritariamente categóricas, ausência de causalidade, baixo poder discriminatório dos thresholds, necessidade de validação futura, caráter experimental do modelo e não substituição da avaliação técnica humana.

## Referências

- Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research.
- Saito, T. e Rehmsmeier, M. (2015). The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets. PLOS ONE.
- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. Monthly Weather Review.
- Kaufman, S., Rosset, S., Perlich, C. e Stitelman, O. (2012). Leakage in Data Mining: Formulation, Detection, and Avoidance. ACM TKDD.
- Documentação do scikit-learn sobre validação de modelos, métricas de classificação, `DummyClassifier`, `OneHotEncoder` e `RandomForestClassifier`.
